# Reddit Web3 ????

?? Notebook ?????? Reddit ?? JSON endpoint ?? Web3 ??????? CSV???? Web3 Knowledge Graph pipeline?

???

- ??? Reddit API key?
- ?? `requests` ???? JSON?
- ?? `pandas` ?????
- ????????? User-Agent ? sleep?
- ????? `pipeline.py` ???

## 1. ???????

?????????? PowerShell ???

```powershell
pip install -r requirements.txt
```

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import time

import pandas as pd
import requests

## 2. ??????

???? Web3 ?? subreddit????

- `ethereum`
- `defi`
- `CryptoCurrency`
- `solana`
- `web3`
- `NFT`

`User-Agent` ????Reddit ???????????????

In [ ]:
SUBREDDITS = ["ethereum", "defi", "CryptoCurrency", "solana", "web3", "NFT"]
SORT = "hot"
POSTS_PER_SUBREDDIT = 10
PAGE_LIMIT = 10
SLEEP_SECONDS = 1.0
USER_AGENT = "web3-kg-coursework/1.0 by your-name"
OUTPUT_PATH = Path("../data/reddit_web3_posts_from_notebook.csv")

## 3. ?????? Reddit JSON

Reddit listing endpoint ???

```text
https://www.reddit.com/r/{subreddit}/{sort}.json
```

?? sort?

- `hot`
- `new`
- `top`
- `rising`

In [ ]:
def fetch_listing(subreddit, sort="hot", limit=25, after=None):
    url = f"https://www.reddit.com/r/{subreddit}/{sort}.json"
    params = {"limit": min(limit, 100)}
    if after:
        params["after"] = after

    response = requests.get(
        url,
        params=params,
        headers={"User-Agent": USER_AGENT},
        timeout=20,
    )
    response.raise_for_status()
    return response.json()

## 4. ? Reddit ?? JSON ?????

Reddit ??? JSON ????????????????????

- `post_id`
- `subreddit`
- `title`
- `selftext`
- `created_utc`
- `score`
- `url`

???? `permalink`?`num_comments`?`author`?`upvote_ratio` ?????

In [ ]:
def parse_post(child):
    data = child.get("data", {})
    created_utc = data.get("created_utc")
    if created_utc:
        created_value = datetime.fromtimestamp(created_utc, tz=timezone.utc).date().isoformat()
    else:
        created_value = ""

    return {
        "post_id": data.get("id", ""),
        "subreddit": data.get("subreddit", ""),
        "title": data.get("title", ""),
        "selftext": data.get("selftext", "") or "",
        "created_utc": created_value,
        "score": data.get("score", 0),
        "url": data.get("url", ""),
        "permalink": "https://www.reddit.com" + data.get("permalink", ""),
        "num_comments": data.get("num_comments", 0),
        "author": data.get("author", ""),
        "upvote_ratio": data.get("upvote_ratio", ""),
        "stickied": bool(data.get("stickied", False)),
    }

## 5. ??????????

?? subreddit ?? `Daily Discussion` ? `Megathread`?????????????????????????????????????

In [ ]:
def is_noise_post(row):
    title = str(row.get("title", "")).lower()
    if row.get("stickied"):
        return True
    noise_terms = [
        "daily discussion",
        "general discussion",
        "weekly discussion",
        "megathread",
        "welcome to",
        "subreddit rules",
    ]
    return any(term in title for term in noise_terms)

## 6. ???? subreddit

??????? Reddit ???? `after`?????????????????

In [ ]:
def crawl_subreddit(subreddit, sort="hot", target_posts=10, page_limit=10, sleep_seconds=1.0):
    rows = []
    after = None

    while len(rows) < target_posts:
        payload = fetch_listing(subreddit, sort=sort, limit=page_limit, after=after)
        listing = payload.get("data", {})
        children = listing.get("children", [])
        if not children:
            break

        for child in children:
            if child.get("kind") == "t3":
                row = parse_post(child)
                if not is_noise_post(row):
                    rows.append(row)
            if len(rows) >= target_posts:
                break

        after = listing.get("after")
        if not after:
            break
        time.sleep(sleep_seconds)

    return rows

## 7. ???? subreddit ???

?? subreddit ??? sleep ?????? Reddit ????

In [ ]:
all_rows = []

for subreddit in SUBREDDITS:
    print(f"Crawling r/{subreddit}...")
    try:
        rows = crawl_subreddit(
            subreddit,
            sort=SORT,
            target_posts=POSTS_PER_SUBREDDIT,
            page_limit=PAGE_LIMIT,
            sleep_seconds=SLEEP_SECONDS,
        )
        print(f"  fetched {len(rows)} posts")
        all_rows.extend(rows)
    except requests.RequestException as exc:
        print(f"  skipped r/{subreddit}: {exc}")
    time.sleep(SLEEP_SECONDS)

posts_df = pd.DataFrame(all_rows).drop_duplicates(subset=["post_id"])
posts_df = posts_df.sort_values(["subreddit", "score"], ascending=[True, False]).reset_index(drop=True)
posts_df.head()

## 8. ?????? subreddit ??

In [ ]:
print("Total posts:", len(posts_df))
posts_df.groupby("subreddit").size().sort_index()

## 9. ?? CSV

?? CSV ?????? Web3 KG pipeline ???

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
posts_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
OUTPUT_PATH

## 10. ????????? Web3 Knowledge Graph

?? PowerShell?? `web3_kg_project` ??????

```powershell
python .\src\pipeline.py --input .\data\reddit_web3_posts_from_notebook.csv --output .\outputs\reddit_live_from_notebook
```

???????

```powershell
python -m http.server 8080
```

???

```text
http://localhost:8080/outputs/reddit_live_from_notebook/interactive_graph.html
```

In [ ]:
# ???? Notebook ?????? pipeline?????? subprocess?
# import subprocess
# subprocess.run([
#     "python", "../src/pipeline.py",
#     "--input", str(OUTPUT_PATH),
#     "--output", "../outputs/reddit_live_from_notebook"
# ], check=True)

## 11. ??????

1. ?????? Reddit?
2. ???? User-Agent?
3. ????????????????
4. ????????????
5. ???????????? Reddit ?? API / PRAW ? OAuth?

## 12. ????

- ?? `new` ? `top` ???
- ????????
- ?????? comment graph?
- ?? spaCy ? LLM ??????????
- ?? Neo4j ? Cypher ??? Bloom ????